### Model Integration with OpenAI, Google Gemini and Groq

#### init

In [ ]:
import os
from dotenv import load_dotenv # 
load_dotenv()

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

#### OpenAI

In [ ]:
from langchain.chat_models import init_chat_model

model=init_chat_model("gpt-4o")
model

response = model.invoke("Hello, how are you?")

response.content

##### OpenAI with dedicated langchain package

In [ ]:
from langchain_openai import ChatOpenAI

# model = ChatOpenAI(
#     model="gemini-2.5-flash",        # Passes target name to LiteLLM
#     openai_api_key="sk-dummy-key",   # Dummy key to satisfy OpenAI SDK
#     openai_api_base="http://localhost:4000"  # LiteLLM proxy endpoint
# )

model = ChatOpenAI(model="gpt-4o")
response = model.invoke("Hello")
response.content

#### Gemini

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash")
response = model.invoke("Hey wassup")
response.content

"Hey there! Not much, just here and ready to help. What's up with you, or what can I do for you today?"

##### Gemini using dedicated langchain lib

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
response = llm.invoke("Hello")
response.content

#### GROQ

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3.6-27b")
response = model.invoke("Hello zere")
response.content

##### Langchain dedicated Groq Library

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile")
response = model.invoke("Hello")
response.content

### Streaming and Batch

#### Streaming

In [ ]:
stream_text = model.stream("Write a 200 word paragraph about why parrots can speak")

for chunk in stream_text:
    print(chunk.text, end="", flush=True)

In [ ]:
for chunk in model.stream("why do parrots wear colorful feathers?"):
    print(chunk.text, end="", flush=True)

#### Batch

Batching a collection of independent requests to a model can significantly improve performance and reduce costs as the processing can be done in parallel.

In [ ]:
responses = model.batch([
    "Hello, how are you?",
    "What is quantum entanglement?",
    "What is the capital of France?"
    ],
    config = {
        "max_concurrency": 5 # the max concurrency tells the moodel how many concuurrent requests it can handle at once e.g. if 10 requests are sent, it will process 5 at a time.
    }
)

for response in responses:
    print(response.content)

### LLM Generation Hyperparameters & Configuration

Understanding and tuning parameters like `temperature`, `top_p`, `top_k`, `max_tokens` / `max_output_tokens`, and `stop_sequences` is essential for controlling model behavior, determinism, cost, and latency.

#### 1. Temperature

`temperature` controls the randomness of token predictions during sampling:
* **Low (0.0 - 0.2)**: Focused, deterministic, reproducible. Ideal for classification, code generation, RAG, and math.
* **Moderate (0.5 - 0.7)**: Balanced coherence and natural variety. Standard default for general Q&A and conversational agents.
* **High (0.8 - 1.5+)**: Creative, diverse, and unconventional. Ideal for brainstorming, storytelling, and divergent ideation.

In [ ]:
# Demonstrating Temperature with ChatGoogleGenerativeAI
from langchain_google_genai import ChatGoogleGenerativeAI

prompt = "Invent a name and a one-sentence slogan for a futuristic quantum coffee shop."

# Deterministic (temperature=0.0)
llm_deterministic = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)
print("--- Temperature = 0.0 (Deterministic) ---")
print(llm_deterministic.invoke(prompt).content)

# Highly Creative (temperature=1.2)
llm_creative = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=1.2)
print("\n--- Temperature = 1.2 (Creative) ---")
print(llm_creative.invoke(prompt).content)

#### 2. Top-P (Nucleus Sampling) and Top-K

* **`top_p` (Nucleus Sampling)**: Dynamically selects tokens from the smallest candidate pool whose cumulative probability mass meets or exceeds `top_p` (e.g., `0.9` keeps the top 90% of probability mass). It cuts off the long tail of unlikely tokens while keeping sampling adaptive.
* **`top_k`**: Restricts the token candidate pool strictly to the `K` highest probability tokens at each generation step (e.g., `top_k=40`), discarding all others regardless of probability distribution.

In [ ]:
# Demonstrating Top-P (Nucleus Sampling) & Top-K candidate pooling
llm_sampling = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.8,
    top_p=0.85,
    top_k=40
)

response_sampling = llm_sampling.invoke("Suggest 3 innovative robotics features for deep-sea underwater exploration.")
print("--- Top-P (0.85) & Top-K (40) Generation ---")
print(response_sampling.content)

#### 3. Max Tokens (`max_tokens` / `max_output_tokens`) & Stop Sequences (`stop_sequences`)

* **`max_output_tokens`**: Caps the maximum number of completion tokens generated. Protects cost budgets and guarantees concise responses.
* **`stop` / `stop_sequences`**: Defines explicit delimiter strings where the LLM immediately stops generating further tokens. Useful for few-shot boundaries, Markdown blocks, or specific turn delimiters.

In [ ]:
# Demonstrating max_output_tokens and stop sequences
llm_constrained = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
    max_output_tokens=40,
    stop=["3."]
)

response_constrained = llm_constrained.invoke("List the steps in the scientific method:\n1. Observation\n2.")
print("--- Constrained Output (Max 40 tokens or halted before '3.') ---")
print("2." + response_constrained.content)

# Dynamic runtime configuration override using .bind()
print("\n--- Runtime Parameter Binding with .bind() ---")
bound_llm = llm.bind(temperature=0.0, max_output_tokens=25)
print(bound_llm.invoke("What is the exact speed of light in m/s? Be succinct.").content)